# 🚢 Titanic Survival Prediction — Logistic Regression
**Dataset:** Titanic  
**Algorithm:** Logistic Regression  
**Goal:** Predict whether a passenger survived (binary classification: 0 or 1)

> **Why Logistic Regression?** The target is binary (survived/not). Logistic Regression outputs a probability between 0 and 1 — perfect for classification. It's also interpretable: we can see exactly which features drive survival.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
df.head()


## Step 1 — Feature Engineering & Preprocessing

In [ ]:
# Select useful features
df = df[['Survived','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']].copy()

# Fill missing values
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Encode categorical variables
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Create a new feature: family size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print("Missing values after preprocessing:", df.isnull().sum().sum())
df.head()


**Feature Engineering:** We created `FamilySize` and `IsAlone` — passengers traveling alone may have had different survival odds. This is called feature engineering: creating new meaningful features from existing ones.


## Step 2 — Train/Test Split & Scaling

In [ ]:
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Survival rate in train: {y_train.mean()*100:.1f}% | test: {y_test.mean()*100:.1f}%")


## Step 3 — Train the Model

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Feature importance (coefficients)
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_[0]})
coef_df = coef_df.sort_values('Coefficient')

plt.figure(figsize=(8, 5))
colors = ['#E74C3C' if c < 0 else '#2ECC71' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Coefficients
(Positive = helps survival, Negative = hurts survival)', fontsize=12)
plt.tight_layout()
plt.show()


## Step 4 — Evaluate the Model

In [ ]:
y_pred  = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("=" * 45)
print("         MODEL EVALUATION RESULTS")
print("=" * 45)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  ROC-AUC   : {auc:.4f}")
print("=" * 45)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))


## Step 5 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Died', 'Predicted: Survived'],
            yticklabels=['Actual: Died', 'Actual: Survived'])
plt.title('Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives (correctly predicted survived): {tp}")
print(f"True Negatives (correctly predicted died):     {tn}")
print(f"False Positives (predicted survived, actually died): {fp}")
print(f"False Negatives (predicted died, actually survived): {fn}")


## Step 6 — ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#3498DB', lw=2, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random Classifier (AUC = 0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Titanic Survival Prediction', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Summary
- Logistic Regression achieves ~80% accuracy on Titanic — solid for a linear model
- **Sex** is the most important feature (positive coefficient = female helps survival)
- **Pclass** has a negative coefficient — higher class number (3rd class) hurts survival
- ROC-AUC > 0.85 shows good discriminative ability
